In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

WINDOW_SIZE     = 30
RUL_CAP         = 125
FAULT_THRESHOLD = 30
BATCH_SIZE      = 64
DROP_SENSORS    = ["s1", "s5", "s10", "s16", "s18", "s19"]
FD_LIST         = ["FD001", "FD002", "FD003", "FD004"]

BASE_DIR       = "/content/drive/MyDrive/TASK-ERAU/C-MAPSS"
FEDAVG_MODEL   = "/content/drive/MyDrive/TASK-ERAU/results/FedAvg/fedavg_global_model.keras"
RQ5_RESULT_DIR = "/content/drive/MyDrive/TASK-ERAU/results/FedAvg_RQ5"
os.makedirs(RQ5_RESULT_DIR, exist_ok=True)

print("Config ready")

✓ Config ready


In [3]:
def load_client_data(fd_name):
    DATA_DIR   = f"{BASE_DIR}/{fd_name}"
    columns    = (["unit", "cycle"]
                  + ["setting_1", "setting_2", "setting_3"]
                  + [f"s{i}" for i in range(1, 22)])

    train_df = pd.read_csv(f"{DATA_DIR}/train_{fd_name}.txt",
                           sep=r"\s+", header=None, names=columns)
    test_df  = pd.read_csv(f"{DATA_DIR}/test_{fd_name}.txt",
                           sep=r"\s+", header=None, names=columns)
    rul_df   = pd.read_csv(f"{DATA_DIR}/RUL_{fd_name}.txt",
                           sep=r"\s+", header=None, names=["RUL"])

    train_df["max_cycle"]   = train_df.groupby("unit")["cycle"].transform("max")
    train_df["RUL_raw"]     = train_df["max_cycle"] - train_df["cycle"]
    train_df["RUL"]         = train_df["RUL_raw"].clip(upper=RUL_CAP)
    train_df["early_fault"] = (train_df["RUL_raw"] <= FAULT_THRESHOLD).astype(int)

    test_rul_map            = dict(zip(range(1, len(rul_df)+1), rul_df["RUL"].values))
    test_df["max_cycle"]    = test_df.groupby("unit")["cycle"].transform("max")
    test_df["final_RUL"]    = test_df["unit"].map(test_rul_map)
    test_df["RUL_raw"]      = test_df["final_RUL"] + (test_df["max_cycle"] - test_df["cycle"])
    test_df["RUL"]          = test_df["RUL_raw"].clip(upper=RUL_CAP)
    test_df["early_fault"]  = (test_df["RUL_raw"] <= FAULT_THRESHOLD).astype(int)

    all_units               = train_df["unit"].unique()
    train_units, val_units  = train_test_split(all_units, test_size=0.2, random_state=SEED)
    train_part              = train_df[train_df["unit"].isin(train_units)].copy()
    val_part                = train_df[train_df["unit"].isin(val_units)].copy()

    all_sensor_cols = [f"s{i}" for i in range(1, 22)]
    feature_cols    = (["setting_1", "setting_2", "setting_3"]
                       + [s for s in all_sensor_cols if s not in DROP_SENSORS])

    N_CONDITIONS     = train_part[["setting_1","setting_2","setting_3"]].nunique().max()
    USE_CLUSTER_NORM = N_CONDITIONS > 2

    if USE_CLUSTER_NORM:
        km = KMeans(n_clusters=6, random_state=SEED, n_init=10)
        train_part["op_cluster"] = km.fit_predict(
            train_part[["setting_1","setting_2","setting_3"]])
        scaler_dict = {}
        for c in range(6):
            mask = train_part["op_cluster"] == c
            sc   = StandardScaler()
            train_part.loc[mask, feature_cols] = sc.fit_transform(
                train_part.loc[mask, feature_cols])
            scaler_dict[c] = sc

        def apply_cluster_scaler(df):
            df = df.copy()
            df["op_cluster"] = km.predict(df[["setting_1","setting_2","setting_3"]])
            for c, sc in scaler_dict.items():
                mask = df["op_cluster"] == c
                if mask.sum() > 0:
                    df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
            return df

        val_part    = apply_cluster_scaler(val_part)
        test_scaled = apply_cluster_scaler(test_df.copy())
    else:
        sc = StandardScaler()
        train_part[feature_cols]  = sc.fit_transform(train_part[feature_cols])
        val_part[feature_cols]    = sc.transform(val_part[feature_cols])
        test_scaled               = test_df.copy()
        test_scaled[feature_cols] = sc.transform(test_scaled[feature_cols])

    def make_windows(df):
        X, y_rul, y_fault, uids, ecycles = [], [], [], [], []
        for unit in sorted(df["unit"].unique()):
            udf = df[df["unit"]==unit].sort_values("cycle").reset_index(drop=True)
            if len(udf) < WINDOW_SIZE:
                continue
            feats  = udf[feature_cols].values
            ruls   = udf["RUL"].values
            faults = udf["early_fault"].values
            cycs   = udf["cycle"].values
            for s in range(len(udf) - WINDOW_SIZE + 1):
                e = s + WINDOW_SIZE
                X.append(feats[s:e]);      y_rul.append(ruls[e-1])
                y_fault.append(faults[e-1]); uids.append(unit)
                ecycles.append(cycs[e-1])
        return (np.array(X, dtype=np.float32), np.array(y_rul, dtype=np.float32),
                np.array(y_fault, dtype=np.float32), np.array(uids), np.array(ecycles))

    X_tr, y_rul_tr, y_fault_tr, uids_tr, ec_tr = make_windows(train_part)
    X_vl, y_rul_vl, y_fault_vl, uids_vl, ec_vl = make_windows(val_part)
    X_te, y_rul_te, y_fault_te, uids_te, ec_te  = make_windows(test_scaled)

    print(f"  {fd_name} | train={X_tr.shape} val={X_vl.shape} test={X_te.shape}")
    return {
        "fd": fd_name, "feature_cols": feature_cols,
        "X_train": X_tr, "y_rul_train": y_rul_tr, "y_fault_train": y_fault_tr,
        "X_val":   X_vl, "y_rul_val":   y_rul_vl, "y_fault_val":   y_fault_vl,
        "X_test":  X_te, "y_rul_test":  y_rul_te, "y_fault_test":  y_fault_te,
        "unit_ids_test": uids_te, "end_cycles_test": ec_te
    }

print("Loading all clients...")
clients = {fd: load_client_data(fd) for fd in FD_LIST}
print("✓ All clients loaded")

Loading all clients...


/tmp/ipykernel_3058/441046711.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.44839748 -1.44839748 -2.09663281 ...  0.49630848  1.79277913
  2.44101445]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
/tmp/ipykernel_3058/441046711.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.80016216 -0.80016216 -1.44839748 -0.15192684 -0.80016216 -0.80016216
 -0.80016216 -1.44839748 -0.80016216  0.49630848 -1.44839748 -0.15192684
 -1.44839748 -0.80016216 -0.15192684  0.49630848 -0.80016216 -0.80016216
 -1.44839748  0.49630848  0.49630848 -0.15192684 -0.15192684 -0.15192684
 -0.80016216 -0.15192684 -0.80016216 -0.15192684 -0.80016216 -0.15192684
  0.49630848  1.14454381  2.44101445 -2.09663281 -0.15192684 -0.15192684
 -0.80

  FD001 | train=(14241, 30, 18) val=(3490, 30, 18) test=(10196, 30, 18)


/tmp/ipykernel_3058/441046711.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.76383951 -1.47678703 -0.76383951 ...  2.08795057  2.08795057
  2.08795057]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
/tmp/ipykernel_3058/441046711.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.18973455 -0.76383951 -1.47678703 ...  2.08795057  2.08795057
  1.37500305]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
/tmp/ipykernel_3058/441046711.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.66205553 -0.76383951 -0.76383951 ..

  FD002 | train=(37432, 30, 18) val=(8787, 30, 18) test=(26505, 30, 18)


/tmp/ipykernel_3058/441046711.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.31207787 -0.31207787 -0.31207787 ...  0.82466606  0.82466606
  2.52978195]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
/tmp/ipykernel_3058/441046711.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.88044983 -0.88044983 -0.88044983 ...  1.96140999  1.96140999
  1.96140999]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
/tmp/ipykernel_3058/441046711.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.88044983 -0.88044983 -1.44882179 ..

  FD003 | train=(17692, 30, 18) val=(4128, 30, 18) test=(13696, 30, 18)


/tmp/ipykernel_3058/441046711.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.27190928 -1.04409768 -1.65800348 ...  1.41152551  1.41152551
  2.0254313 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
/tmp/ipykernel_3058/441046711.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.79761971 -0.43019189  0.18371391 ... -0.43019189  0.79761971
  0.79761971]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
/tmp/ipykernel_3058/441046711.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.43019189 -1.65800348 -1.65800348 ..

  FD004 | train=(43523, 30, 18) val=(10505, 30, 18) test=(34081, 30, 18)
✓ All clients loaded


In [4]:
def weighted_bce(pos_weight):
    def loss_fn(y_true, y_pred):
        bce    = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        weight = y_true * pos_weight + (1 - y_true) * 1.0
        return tf.reduce_mean(weight * bce)
    return loss_fn

# load with dummy pos_weight — weights are already baked in
global_model = tf.keras.models.load_model(
    FEDAVG_MODEL,
    custom_objects={"loss_fn": weighted_bce(1.0)}
)
print("✓ FedAvg global model loaded")
print(f"  Params: {global_model.count_params():,}")

✓ FedAvg global model loaded
  Params: 221,378


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 66 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
def compute_pos_weight(y_fault):
    neg = np.sum(y_fault == 0)
    pos = np.sum(y_fault == 1)
    return float(np.sqrt(neg / pos)) if pos > 0 else 1.0

def build_model(input_shape, pos_weight):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(64,  kernel_size=5, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    rul_output   = layers.Dense(1, activation="linear",  name="rul_output")(x)
    fault_output = layers.Dense(1, activation="sigmoid", name="fault_output")(x)
    model = models.Model(inputs=inputs, outputs=[rul_output, fault_output])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={"rul_output": "mse", "fault_output": weighted_bce(pos_weight)},
        loss_weights={"rul_output": 1.0, "fault_output": 1.5},
        metrics={"rul_output": ["mae"],
                 "fault_output": ["accuracy",
                                  tf.keras.metrics.AUC(curve="PR", name="auprc")]}
    )
    return model

# train one local model per client starting from global weights
input_shape   = (WINDOW_SIZE, len(clients["FD001"]["feature_cols"]))
local_models  = {}

print("Fine-tuning one local model per client from global weights...")
for fd, client in clients.items():
    pw    = compute_pos_weight(client["y_fault_train"])
    model = build_model(input_shape, pw)
    model.set_weights(global_model.get_weights())   # start from global

    model.fit(
        client["X_train"],
        {"rul_output":   client["y_rul_train"] / RUL_CAP,
         "fault_output": client["y_fault_train"]},
        validation_data=(
            client["X_val"],
            {"rul_output":   client["y_rul_val"] / RUL_CAP,
             "fault_output": client["y_fault_val"]}),
        epochs=5, batch_size=BATCH_SIZE, verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True)]
    )
    local_models[fd] = model
    print(f"  ✓ {fd} local model ready")

print("\n✓ All local models trained")

Fine-tuning one local model per client from global weights...
  ✓ FD001 local model ready
  ✓ FD002 local model ready
  ✓ FD003 local model ready
  ✓ FD004 local model ready

✓ All local models trained


In [6]:
def get_auprc(model, client):
    """Score model on client's test set — returns AUPRC."""
    _, pred_fault_prob = model.predict(client["X_test"], verbose=0)
    pred_fault_prob    = pred_fault_prob.flatten()
    try:
        return average_precision_score(client["y_fault_test"], pred_fault_prob)
    except:
        return 0.0

# build 4x4 cross-evaluation matrix
# cross_matrix[i][j] = AUPRC of client_i model evaluated on client_j data
print("Building cross-evaluation matrix (4×4)...")
n          = len(FD_LIST)
cross_matrix = np.zeros((n, n))

for i, fd_model in enumerate(FD_LIST):
    for j, fd_data in enumerate(FD_LIST):
        score = get_auprc(local_models[fd_model], clients[fd_data])
        cross_matrix[i][j] = score
        print(f"  Model {fd_model} → Data {fd_data}: AUPRC={score:.4f}")

print("\nCross-evaluation matrix (rows=model, cols=data):")
cross_df = pd.DataFrame(cross_matrix, index=FD_LIST, columns=FD_LIST)
print(cross_df.round(4))

# ── plot heatmap ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cross_df, annot=True, fmt=".3f", cmap="YlOrRd_r",
            vmin=0.5, vmax=1.0, ax=ax,
            linewidths=0.5, linecolor="gray")
ax.set_xlabel("Evaluated on (data)")
ax.set_ylabel("Model from (client)")
ax.set_title("RQ5 — Cross-evaluation AUPRC matrix\n"
             "Diagonal = self-evaluation | Off-diagonal = cross-client scoring")
plt.tight_layout()
plt.savefig(os.path.join(RQ5_RESULT_DIR, "rq5_cross_eval_matrix.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("✓ Cross-evaluation matrix saved")

Building cross-evaluation matrix (4×4)...
  Model FD001 → Data FD001: AUPRC=0.9058
  Model FD001 → Data FD002: AUPRC=0.7813
  Model FD001 → Data FD003: AUPRC=0.8512
  Model FD001 → Data FD004: AUPRC=0.6175
  Model FD002 → Data FD001: AUPRC=0.8858
  Model FD002 → Data FD002: AUPRC=0.8239
  Model FD002 → Data FD003: AUPRC=0.7019
  Model FD002 → Data FD004: AUPRC=0.6803
  Model FD003 → Data FD001: AUPRC=0.8827
  Model FD003 → Data FD002: AUPRC=0.7549
  Model FD003 → Data FD003: AUPRC=0.8789
  Model FD003 → Data FD004: AUPRC=0.6453
  Model FD004 → Data FD001: AUPRC=0.8314
  Model FD004 → Data FD002: AUPRC=0.7351
  Model FD004 → Data FD003: AUPRC=0.7125
  Model FD004 → Data FD004: AUPRC=0.6165

Cross-evaluation matrix (rows=model, cols=data):
        FD001   FD002   FD003   FD004
FD001  0.9058  0.7813  0.8512  0.6175
FD002  0.8858  0.8239  0.7019  0.6803
FD003  0.8827  0.7549  0.8789  0.6453
FD004  0.8314  0.7351  0.7125  0.6165
✓ Cross-evaluation matrix saved


In [7]:
def compute_mmd(X_a, X_b, n_samples=500):
    """
    Maximum Mean Discrepancy between two datasets.
    Flatten windows to 2D, subsample for speed.
    Higher MMD = more different distributions.
    """
    # flatten: (N, window, features) → (N, window*features)
    Xa = X_a.reshape(len(X_a), -1)
    Xb = X_b.reshape(len(X_b), -1)

    # subsample
    idx_a = np.random.choice(len(Xa), min(n_samples, len(Xa)), replace=False)
    idx_b = np.random.choice(len(Xb), min(n_samples, len(Xb)), replace=False)
    Xa    = Xa[idx_a]
    Xb    = Xb[idx_b]

    # RBF kernel MMD
    def rbf_kernel(X, Y, gamma=1.0):
        XX = np.sum(X**2, axis=1, keepdims=True)
        YY = np.sum(Y**2, axis=1, keepdims=True)
        XY = X @ Y.T
        dist = XX + YY.T - 2 * XY
        return np.exp(-gamma * dist)

    K_aa = rbf_kernel(Xa, Xa).mean()
    K_bb = rbf_kernel(Xb, Xb).mean()
    K_ab = rbf_kernel(Xa, Xb).mean()
    return float(K_aa + K_bb - 2 * K_ab)

# build 4x4 MMD distance matrix
print("Computing distributional distance matrix (MMD)...")
dist_matrix = np.zeros((n, n))

for i, fd_a in enumerate(FD_LIST):
    for j, fd_b in enumerate(FD_LIST):
        if i == j:
            dist_matrix[i][j] = 0.0
        elif j > i:
            mmd = compute_mmd(
                clients[fd_a]["X_train"],
                clients[fd_b]["X_train"]
            )
            dist_matrix[i][j] = mmd
            dist_matrix[j][i] = mmd   # symmetric
        print(f"  MMD({fd_a}, {fd_b}) = {dist_matrix[i][j]:.4f}")

dist_df = pd.DataFrame(dist_matrix, index=FD_LIST, columns=FD_LIST)
print("\nDistributional distance matrix (MMD):")
print(dist_df.round(4))

# ── plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(dist_df, annot=True, fmt=".3f", cmap="Blues",
            ax=ax, linewidths=0.5, linecolor="gray")
ax.set_title("RQ5 — Distributional distance matrix (MMD)\n"
             "Higher = more different distributions")
plt.tight_layout()
plt.savefig(os.path.join(RQ5_RESULT_DIR, "rq5_mmd_distance_matrix.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("✓ MMD matrix saved")

Computing distributional distance matrix (MMD)...
  MMD(FD001, FD001) = 0.0000
  MMD(FD001, FD002) = 0.0040
  MMD(FD001, FD003) = 0.0040
  MMD(FD001, FD004) = 0.0040
  MMD(FD002, FD001) = 0.0040
  MMD(FD002, FD002) = 0.0000
  MMD(FD002, FD003) = 0.0040
  MMD(FD002, FD004) = 0.0040
  MMD(FD003, FD001) = 0.0040
  MMD(FD003, FD002) = 0.0040
  MMD(FD003, FD003) = 0.0000
  MMD(FD003, FD004) = 0.0040
  MMD(FD004, FD001) = 0.0040
  MMD(FD004, FD002) = 0.0040
  MMD(FD004, FD003) = 0.0040
  MMD(FD004, FD004) = 0.0000

Distributional distance matrix (MMD):
       FD001  FD002  FD003  FD004
FD001  0.000  0.004  0.004  0.004
FD002  0.004  0.000  0.004  0.004
FD003  0.004  0.004  0.000  0.004
FD004  0.004  0.004  0.004  0.000
✓ MMD matrix saved


In [8]:
def compute_aggregation_weights(cross_matrix, dist_matrix, beta=0.5):
    """
    Standard weights: row mean of cross_matrix (how well each model scores on others)
    Corrected weights: adjust by distributional distance

    beta: 0=pure score-based, 1=fully distance-corrected
    """
    n = len(FD_LIST)

    # ── standard weights (ref [10] approach) ─────────────────────────────────
    # each client scores other clients' models — column mean
    # (how well does each model perform across all clients)
    standard_scores = cross_matrix.mean(axis=1)   # mean AUPRC per model
    standard_weights = standard_scores / standard_scores.sum()

    # ── distance-corrected weights ────────────────────────────────────────────
    # penalty: if a model scores low on a client that is very different,
    # that low score is less informative — discount it
    corrected_scores = np.zeros(n)
    for i in range(n):       # model i
        weighted_score = 0.0
        total_weight   = 0.0
        for j in range(n):   # data j
            if i == j:
                continue
            # discount cross-score by distributional distance
            # higher distance → lower confidence in that score
            dist_penalty  = 1.0 / (1.0 + dist_matrix[i][j])
            weighted_score += cross_matrix[i][j] * dist_penalty
            total_weight   += dist_penalty
        corrected_scores[i] = weighted_score / total_weight if total_weight > 0 else 0

    # blend standard and corrected
    final_scores  = (1 - beta) * standard_scores + beta * corrected_scores
    corrected_weights = final_scores / final_scores.sum()

    return standard_weights, corrected_weights, standard_scores, corrected_scores


standard_weights, corrected_weights, std_scores, corr_scores = \
    compute_aggregation_weights(cross_matrix, dist_matrix, beta=0.5)

print("=" * 55)
print("RQ5 — Aggregation Weight Comparison")
print("=" * 55)
print(f"\n{'Client':<8} {'Std Score':<12} {'Std Weight':<13} "
      f"{'Corr Score':<13} {'Corr Weight':<12} {'Change'}")
print("-" * 70)
for i, fd in enumerate(FD_LIST):
    change = corrected_weights[i] - standard_weights[i]
    arrow  = "↑" if change > 0 else "↓"
    print(f"{fd:<8} {std_scores[i]:<12.4f} {standard_weights[i]:<13.4f} "
          f"{corr_scores[i]:<13.4f} {corrected_weights[i]:<12.4f} "
          f"{arrow}{abs(change):.4f}")

# ── plot: weight comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(FD_LIST))

axes[0].bar(x - 0.2, standard_weights,  0.35, label="Standard (biased)",   alpha=0.8, color="steelblue")
axes[0].bar(x + 0.2, corrected_weights, 0.35, label="Corrected (RQ5)",     alpha=0.8, color="darkorange")
axes[0].set_xticks(x); axes[0].set_xticklabels(FD_LIST)
axes[0].set_ylabel("Aggregation weight")
axes[0].set_title("Standard vs Corrected aggregation weights")
axes[0].legend(); axes[0].grid(alpha=0.3)

# diagonal (self-score) vs off-diagonal mean per client
self_scores    = np.diag(cross_matrix)
off_diag_means = (cross_matrix.sum(axis=1) - self_scores) / (n - 1)

axes[1].bar(x - 0.2, self_scores,    0.35, label="Self-score",        alpha=0.8, color="green")
axes[1].bar(x + 0.2, off_diag_means, 0.35, label="Cross-score mean",  alpha=0.8, color="tomato")
axes[1].set_xticks(x); axes[1].set_xticklabels(FD_LIST)
axes[1].set_ylabel("AUPRC")
axes[1].set_title("Self-score vs Cross-score mean\n(gap = distributional bias)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("RQ5 — Non-IID Validation Bias Analysis", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RQ5_RESULT_DIR, "rq5_weight_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── save full report ──────────────────────────────────────────────────────────
lines = [
    "RQ5 — Non-IID Validation Bias\n",
    "=" * 55 + "\n\n",
    "Cross-evaluation AUPRC matrix (rows=model, cols=data):\n",
    cross_df.round(4).to_string() + "\n\n",
    "Distributional distance matrix (MMD):\n",
    dist_df.round(4).to_string() + "\n\n",
    "Aggregation weight comparison:\n",
    f"{'Client':<8} {'Std Weight':<14} {'Corr Weight':<14} {'Change'}\n",
    "-" * 45 + "\n"
]
for i, fd in enumerate(FD_LIST):
    change = corrected_weights[i] - standard_weights[i]
    arrow  = "↑" if change > 0 else "↓"
    lines.append(f"{fd:<8} {standard_weights[i]:<14.4f} "
                 f"{corrected_weights[i]:<14.4f} {arrow}{abs(change):.4f}\n")

lines += [
    "\n" + "=" * 55 + "\n",
    "\nKey finding:\n",
    f"  Largest weight change: "
    f"{FD_LIST[np.argmax(np.abs(corrected_weights - standard_weights))]}\n",
    f"  Max bias (self vs cross score gap): "
    f"{(self_scores - off_diag_means).max():.4f} on "
    f"{FD_LIST[np.argmax(self_scores - off_diag_means)]}\n"
]

with open(os.path.join(RQ5_RESULT_DIR, "rq5_report.txt"), "w") as f:
    f.writelines(lines)

print(f"\n✓ All RQ5 results saved to: {RQ5_RESULT_DIR}")
print("  rq5_cross_eval_matrix.png")
print("  rq5_mmd_distance_matrix.png")
print("  rq5_weight_comparison.png")
print("  rq5_report.txt")

RQ5 — Aggregation Weight Comparison

Client   Std Score    Std Weight    Corr Score    Corr Weight  Change
----------------------------------------------------------------------
FD001    0.7890       0.2565        0.7500        0.2522       ↓0.0043
FD002    0.7730       0.2513        0.7560        0.2505       ↓0.0007
FD003    0.7904       0.2569        0.7609        0.2542       ↓0.0027
FD004    0.7239       0.2353        0.7597        0.2431       ↑0.0078

✓ All RQ5 results saved to: /content/drive/MyDrive/TASK-ERAU/results/FedAvg_RQ5
  rq5_cross_eval_matrix.png
  rq5_mmd_distance_matrix.png
  rq5_weight_comparison.png
  rq5_report.txt
